## INIT

In [0]:
import sys
import os
sys.path.append(os.path.abspath('../../'))
from utils.logger_utils import get_project_logger

logger = get_project_logger("silver_Layer_private")
logger.info("Success! The logger is working.")

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType
from datetime import datetime
from utils.pipeline_monitoring import log_pipeline_run

dataset_name = "private_vehicles"
layer = "silver"

start_time = datetime.now()

## Reading from bronze layer

In [0]:
logger.info("Step 1/5: Reading data from Bronze layer...")
df_bronze = spark.table("transport_lakehouse.bronze.car_plate_numbers_raw")
logger.info(f"Successfully read {df_bronze.count()} rows from Bronze.")

## Columns organization and new column aliases

In [0]:
logger.info("Step 2/5: Starting column reorganization and aliasing...")
df_silver = df_bronze.select(
  F.col("mispar_rechev").cast("int").alias("car_num"),
  F.col("kinuy_mishari").cast("string").alias("commercial_nm"),
  F.col("sug_delek_nm").cast("string").alias("fuel_type_nm"),
  F.col("tozeret_cd").cast("int").alias("manufacturer_cd"),
  F.col("tozeret_nm").cast("string").alias("manufacturer_nm"),
  F.col("tzeva_cd").cast("int").alias("color_cd"),
  F.col("tzeva_rechev").cast("string").alias("color_nm"),
  F.col("baalut").cast("string").alias("ownership"),
  F.col("shnat_yitzur").cast("int").alias("prod_year"),
  F.col("moed_aliya_lakvish").cast("date").alias("road_entry_dt"),
  F.col("mivchan_acharon_dt").cast("date").alias("last_test_dt"),
  F.col("tokef_dt").cast("date").alias("test_valid_until_dt")
)

logger.info("Step 2/5: Finished column reorganization.")


## Null handling

In [0]:
logger.info("Step 3/5: Starting NULL handling for 'road_entry_dt'...")
df_silver = df_silver.withColumn(
    "road_entry_dt",
    F.coalesce(
        F.col("road_entry_dt"),
        F.to_date(
            F.concat(F.col("prod_year").cast("string"), F.lit("-01-01")),
            "yyyy-MM-dd"
        )
    )
)

logger.info("Step 3/5: Finished NULL handling.")

## String Fixes

In [0]:
logger.info("Step 4/5: Starting string fixes (Manufacturer names)...")
df_silver = df_silver.withColumn(
    "manufacturer_nm",
    F.when(
        F.col("manufacturer_cd") == 593,
        "מרצדס בנץ גרמניה"
    ).otherwise(F.col("manufacturer_nm"))
)

logger.info("Step 4/5: Finished string fixes.")


## Count final rows for monitoring

In [0]:
row_count = df_silver.count()
logger.info(f"Final Silver row count: {row_count}")

## Writing to silver layer

In [0]:
try:
    target_table = "transport_lakehouse.silver.silver_vehicles_private"
    logger.info(f"Step 5/5: Writing final DataFrame to Silver table: {target_table}")
    (
        df_silver.write 
            .mode("overwrite") 
            .format("delta") 
            .option("overwriteSchema", "true")
            .saveAsTable(target_table)

    )
    end_time = datetime.now()

    log_pipeline_run(
        spark=spark,
        dataset_name=dataset_name,
        layer=layer,
        run_start_time=start_time,
        run_end_time=end_time,
        status="SUCCESS",
        rows_ingested=row_count,
        error_message=None
    )

    logger.info("--- Silver_Vehicles_Private Process Completed Successfully ---")

except Exception as e:
    end_time = datetime.now()
    logger.error(f"[SILVER FAILED] dataset=private_vehicles error={e}")

    log_pipeline_run(
        spark=spark,
        dataset_name=dataset_name,
        layer="silver",
        run_start_time=start_time,
        run_end_time=end_time,
        status="FAILED",
        rows_ingested=0,
        error_message=str(e)
    )

    raise
